# PROYECTO: Piloto Validacion PRs
# OBJETIVO: Cargar saldos diarios de clientes a Silver
# VERSION: 1.0.0
# DESARROLLADOR: Jose Fajardo
# FECHA: 2026-07-31


In [ ]:
import logging

from pyspark.sql import functions as F


In [ ]:
def get_logger(nombre):
    """Devuelve el logger estandar del proceso."""
    log = logging.getLogger(nombre)
    log.setLevel(logging.INFO)
    return log


def read_saldos(catalogo, fecha):
    """Lee los saldos filtrados por fecha de proceso."""
    return (
        spark.table(catalogo + ".mmff.h_saldo")
        .select("cod_cliente", "mto_saldo", "fec_proceso")
        .filter(F.col("fec_proceso") == fecha)
    )


def write_saldos(df_origen, destino):
    """Escribe el resultado en formato Delta."""
    df_origen.write.format("delta").mode("overwrite").saveAsTable(destino)


In [ ]:
dbutils.widgets.text("catalogo", "", "Catalogo")
dbutils.widgets.text("fecha_proceso", "", "Fecha de proceso")

var_catalogo = dbutils.widgets.get("catalogo")
var_fecha_proceso = dbutils.widgets.get("fecha_proceso")


In [ ]:
NOMBRE_PROCESO = "carga_saldos_diaria"
tabla_destino = var_catalogo + ".mmff.h_saldo_cliente"


In [ ]:
logger = get_logger(NOMBRE_PROCESO)
logger.info("Inicio del proceso %s", NOMBRE_PROCESO)
logger.info("Parametros recibidos: catalogo=%s fecha=%s", var_catalogo, var_fecha_proceso)

df_saldos = read_saldos(var_catalogo, var_fecha_proceso)
df_vigentes = spark.sql(
    "SELECT cod_cliente, mto_saldo FROM mb_silver_prod.mmff.h_saldo WHERE mto_saldo > 0"
)

write_saldos(df_saldos, tabla_destino)
logger.info("Fin del proceso %s", NOMBRE_PROCESO)
